In [ ]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [1]:
# Check GPU status
!nvidia-smi

Sat Apr 25 21:17:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.86.16              Driver Version: 570.86.16      CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:03:00.0 Off |                  N/A |
|  0%   36C    P8             18W /  350W |       2MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%capture
# For Qwen3-VL
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.57.1
%pip install --no-deps trl==0.22.2

# Configuration

In [6]:
SEED = 42

# Model configuration
MODEL_ID = 'alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K-v2'
RESUME_MODEL_ID = None
RESUME_CKPT_STEP = None

# Data configuration
SFT_DATA_ID = 'alxxtexxr/coco_captions_1.25k_serialized_for_Qwen3-VL-8B-Thinking-VCB8K'
SFT_COT_DATA_ID = 'alxxtexxr/R1-Onevision-1.25K'

# Training configuration
MINI_BATCH_SIZE = 2
GRAD_ACC_STEPS = 16
NUM_EPOCHS = 20
WARMUP_STEPS = 20
LR = 2e-4

# NOTE: 
# For batch_size = 32
# - steps_per_epoch = train_size / batch_size = 2000 / 32 = 125 ~> 126
# - warmup_steps    = steps_per_epoch * min_num_epochs * warmup_ratio = 126 * 3 * 0.05 = 18.9 ~> 20
# For batch_size = 16
# - steps_per_epoch = train_size / batch_size = 2000 / 16 = 250 ~> 250
# - warmup_steps    = steps_per_epoch * min_num_epochs * warmup_ratio = 250 * 3 * 0.05 = 37.5 ~> 40

In [8]:
from datetime import datetime

# Resume training configuration
resume_from_checkpoint = bool(RESUME_MODEL_ID)
if resume_from_checkpoint:
    model_name = RESUME_MODEL_ID
    hub_model_id = RESUME_MODEL_ID
    
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
    if RESUME_CKPT_STEP:
        resume_from_checkpoint = f"{hub_model_id}/checkpoint-{RESUME_CKPT_STEP}"
        # Ensure the checkpoint exists
        assert os.path.exists(resume_from_checkpoint), f"Checkpoint {resume_from_checkpoint} does not exist!"
            
else:
    # TODO: Handle the Hugging Face username properly
    model_name = MODEL_ID
    hub_model_id = (
        f'{MODEL_ID.split('-v')[0] if '-v' in MODEL_ID else MODEL_ID}'
        f'-SFT-QLoRA-v{datetime.now().strftime("%y%m%d%H%M%S")}'
    )
project_name = hub_model_id.split('/')[-1]

print("Resume from checkpoint:", resume_from_checkpoint)
print("Model name:", model_name)
print("Project name:", project_name)
print("Hugging Face model ID:", hub_model_id)

Resume from checkpoint: False
Model name: alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K-v2
Project name: Qwen3-VL-8B-Thinking-VCB8K-SFT-QLoRA-v260425212332
Hugging Face model ID: alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K-SFT-QLoRA-v260425212332


In [9]:
import os
from unsloth import FastVisionModel
import random
import numpy as np
import torch
import transformers

def set_seed(seed, verbose=True):
    # Set random seed for os
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'  # CUDA deterministic behavior (11.2+)

    # Set random seed for NumPy
    np.random.seed(seed)

    # Set random seed for Torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)           # If using multi-GPU
    # torch.use_deterministic_algorithms(True) # ERROR!
    torch.backends.cudnn.deterministic = True  # Ensures deterministic results
    torch.backends.cudnn.benchmark = False     # Avoids non-deterministic algorithms

    # Set random seed for Transformers
    transformers.set_seed(seed)

    # Set random seed for sklearn and Python's own random module
    random.seed(seed) 
    
    if verbose:
        print("Set random seed:", seed)

set_seed(SEED)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Set random seed: 42


In [10]:
# Fix TorchDynamo issue
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

# Fix Unsloth issue
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ''

# Model

In [11]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = resume_from_checkpoint if isinstance(resume_from_checkpoint, str) else model_name,
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16-bit LoRA.
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    # dtype = torch.float16, # Force FP16
)

==((====))==  Unsloth 2026.4.8: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.568 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/5.99G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.92G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/5.76G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/13.0M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

In [12]:
tokenizer.tokenizer.encode("<vthink><|vtok_4167|><|vtok_4384|></vthink><think></think><answer></answer>")

[151669, 155840, 156057, 151670, 151667, 151668, 151671, 151672]

In [13]:
from peft import PeftModelForCausalLM

if not isinstance(model, PeftModelForCausalLM):
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers     = False, # False if not finetuning vision layers
        finetune_language_layers   = True,  # False if not finetuning language layers
        finetune_attention_modules = True,  # False if not finetuning attention layers
        finetune_mlp_modules       = True,  # False if not finetuning MLP layers

        r = 16,          # The larger, the higher the accuracy, but might overfit
        lora_alpha = 16, # Recommended alpha == r at least
        lora_dropout = 0,
        bias = 'none',
        random_state = SEED,
        use_rslora = False,  # We support rank stabilized LoRA
        loftq_config = None, # And LoftQ
        # target_modules = 'all-linear', # Optional now! Can specify a list if needed
    )
else:   
    print("Model is already a PeftModelForCausalLM, skipping PEFT wrapping.")

# Data

In [14]:
# Load main train and set sets
from datasets import load_dataset

dataset = load_dataset(SFT_DATA_ID)
train_set = dataset['train']
val_set = dataset['val']

print("Train set:")
print(train_set)
print("\nValidation set:")
print(val_set)

README.md:   0%|          | 0.00/627 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/389M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/52.1M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/49.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/125 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/125 [00:00<?, ? examples/s]

Train set:
Dataset({
    features: ['filename', 'cocoid', 'caption', 'image', 'image_serialized'],
    num_rows: 1000
})

Validation set:
Dataset({
    features: ['filename', 'cocoid', 'caption', 'image', 'image_serialized'],
    num_rows: 125
})


In [15]:
VTHINK_START = "<vthink>"
VTHINK_END = "</vthink>"

def convert_to_conversation(sample):
    conversation = [
        { 
            "role": "user",
            "content" : [
                {
                    "type" : "text",  
                    # "text": f"Description: {sample['caption']}\n\nBased on the description, provide the visual representations between {VTHINK_START} and {VTHINK_END}",
                    # "text": f"{sample['caption']}\n\nBased on the description above, provide the visual representations between {VTHINK_START} and {VTHINK_END}",
                    "text": f"Generate visual representations for this description between {VTHINK_START} and {VTHINK_END}.\n\nDescription: {sample['caption']}",
                },
                # {"type" : "image", "image" : sample["image"]},
            ],
        },
        { 
            "role" : "assistant",
            "content" : [
                {"type" : "text",  "text"  : f"<vthink>{sample['image_serialized']}</vthink>"},
            ],
        },
    ]
    return { "messages" : conversation }

train_dataset = [convert_to_conversation(sample) for sample in train_set]
val_dataset = [convert_to_conversation(sample) for sample in val_set]

print("Train dataset sample:")
print(train_dataset[0])
print("\nValidation dataset sample:")
print(val_dataset[0])

Train dataset sample:
{'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'Generate visual representations for this description between <vthink> and </vthink>.\n\nDescription: A man wearing a glove with a bird on top of it.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': '<vthink><|vtok_7550|><|vtok_974|><|vtok_4167|><|vtok_4384|><|vtok_4384|><|vtok_4384|><|vtok_4384|><|vtok_6079|><|vtok_4384|><|vtok_2710|><|vtok_2710|><|vtok_7983|><|vtok_2710|><|vtok_3179|><|vtok_2652|><|vtok_4280|><|vtok_6980|><|vtok_3407|><|vtok_6678|><|vtok_5357|><|vtok_3221|><|vtok_6079|><|vtok_4978|><|vtok_5639|><|vtok_3631|><|vtok_5062|><|vtok_6079|><|vtok_2710|><|vtok_7567|><|vtok_4143|><|vtok_1177|><|vtok_1384|><|vtok_5091|><|vtok_6152|><|vtok_4154|><|vtok_5949|><|vtok_4384|><|vtok_6637|><|vtok_3909|><|vtok_6079|><|vtok_6079|><|vtok_6079|><|vtok_7874|><|vtok_4154|><|vtok_1812|><|vtok_5357|><|vtok_6234|><|vtok_6555|><|vtok_7917|><|vtok_5683|><|vtok_7654|><|vtok_6079|><|vtok_4384|

# CoT Data

In [16]:
# Load CoT train and val sets
cot_dataset = load_dataset(SFT_COT_DATA_ID)
cot_train_set = cot_dataset['train']
cot_val_set = cot_dataset['validation']

# Resize images to (512, 512)
def resize_images(example):
    image = example["image"]
    image = image.resize((512, 512))
    example["image"] = image
    return example
cot_train_set = cot_train_set.map(resize_images)
cot_val_set = cot_val_set.map(resize_images)

# Convert images to RGB
def convert_to_rgb(example):
    image = example["image"]
    if image.mode != "RGB":
        image = image.convert("RGB")
    example["image"] = image
    return example
cot_train_set = cot_train_set.map(convert_to_rgb)
cot_val_set = cot_val_set.map(convert_to_rgb)

print("CoT train set:")
print(cot_train_set)
print("\nCoT validation set:")
print(cot_val_set)

README.md:   0%|          | 0.00/612 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/119M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/11.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/15.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

CoT train set:
Dataset({
    features: ['id', 'image', 'instruction', 'response'],
    num_rows: 1000
})

CoT validation set:
Dataset({
    features: ['id', 'image', 'instruction', 'response'],
    num_rows: 125
})


In [17]:
def convert_cot_sample_to_conversation(sample):
    conversation = [
        { 
            "role": "user",
            "content" : [
                {
                    "type" : "text",
                    "text": sample["instruction"],
                },
                {"type" : "image", "image" : sample["image"]},
            ],
        },
        { 
            "role" : "assistant",
            "content" : [
                {
                    "type" : "text",
                    "text"  : sample["response"],
                },
            ],
        },
    ]
    return { "messages" : conversation }

cot_train_dataset = [convert_cot_sample_to_conversation(sample) for sample in cot_train_set]
cot_val_dataset = [convert_cot_sample_to_conversation(sample) for sample in cot_val_set]

print("CoT train dataset sample:")
print(cot_train_dataset[0])
print("\nCoT validation dataset sample:")
print(cot_val_dataset[0])

CoT train dataset sample:
{'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'According to the given food web, which animal would suffer as a result of drying up of grass in summer?plankton\nfungi\nturtle\nants\nAnswer with the given options directly.\nProvide your final answer between <answer> and </answer>.'}, {'type': 'image', 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=512x512 at 0x742060C62300>}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': "<think>\nOkay, so I need to figure out which animal would suffer if the grass dries up in the summer based on the given food web. Let me start by analyzing the image itself.\n\nThe image shows a complex food web with various organisms connected by arrows indicating energy flow. The sun is the primary energy source, and the primary producers include grass, algae, lily pads, and a tree. Grass receives energy directly from the sun, so it's a primary producer.\n\nLooking at the primary consumers, 

In [18]:
# Combine the main and CoT dataset, and tag them with 'is_cot' for later balanced sampling
from datasets import Dataset

def tag_cot(dataset, is_cot):
    for x in dataset:
        x['is_cot'] = is_cot
    return dataset

train_dataset = Dataset.from_list(tag_cot(train_dataset, False) + tag_cot(cot_train_dataset, True))
eval_dataset  = Dataset.from_list(tag_cot(val_dataset, False) + tag_cot(cot_val_dataset, True))

# Training

In [19]:
import random
from torch.utils.data import Sampler
from trl import SFTTrainer

class BalancedBatchSampler(Sampler):
    def __init__(self, dataset, batch_size):
        assert batch_size % 2 == 0
        self.dataset = dataset
        self.batch_size = batch_size
        self.half = batch_size // 2

        self.reg_indices = [
            i for i, x in enumerate(dataset) if not x['is_cot']
        ]
        self.cot_indices = [
            i for i, x in enumerate(dataset) if x['is_cot']
        ]

    def __iter__(self):
        random.shuffle(self.reg_indices)
        random.shuffle(self.cot_indices)

        reg_ptr, cot_ptr = 0, 0

        while reg_ptr + self.half <= len(self.reg_indices) and \
              cot_ptr + self.half <= len(self.cot_indices):

            batch = (
                self.reg_indices[reg_ptr:reg_ptr+self.half] +
                self.cot_indices[cot_ptr:cot_ptr+self.half]
            )

            random.shuffle(batch)
            yield batch

            reg_ptr += self.half
            cot_ptr += self.half

    def __len__(self):
        return min(len(self.reg_indices), len(self.cot_indices)) // self.half

class BalancedSFTTrainer(SFTTrainer):
    def get_train_dataloader(self):
        sampler = BalancedBatchSampler(
            self.train_dataset,
            self.args.per_device_train_batch_size
        )

        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_sampler=sampler,
            collate_fn=self.data_collator,
        )

In [22]:
import math
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTConfig
from transformers import EarlyStoppingCallback

FastVisionModel.for_training(model) # Enable for training

max_steps = math.ceil(len(train_dataset) / (MINI_BATCH_SIZE * GRAD_ACC_STEPS)) * NUM_EPOCHS
print("Calculated max steps:", max_steps)

trainer = BalancedSFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        # Training arguments
        seed = SEED,
        
        per_device_train_batch_size = MINI_BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACC_STEPS,
        
        # max_steps = 50, # For testing
        max_steps = max_steps,
        # num_train_epochs = NUM_EPOCHS,
        # warmup_ratio=0.05,
        warmup_steps = WARMUP_STEPS,
        learning_rate = LR,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        max_grad_norm=1.0,
        weight_decay = 0.01,
        
        # Validation arguments
        eval_strategy='steps',
        eval_steps=20,
        
        # Logging arguments
        logging_strategy='steps',
        logging_steps=10,
        # logging_first_step=True,
        report_to=['tensorboard', 'wandb'],
        
        # Saving arguments
        save_strategy='steps',
        # save_steps=1, # For testing
        save_steps=20,
        # save_total_limit=5, # 1 best + 4 recent checkpoints. Warning: It doesn't work
        
        # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
        # So you will find one checkpoint at the end of each epoch.
        # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
        load_best_model_at_end=True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,

        output_dir=project_name,
        hub_model_id=hub_model_id,
        push_to_hub=True,
        hub_strategy='all_checkpoints',
        hub_always_push=True,

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 1024,
    ),
    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience = 2,
            early_stopping_threshold = 0.001,
        )
    ]
)

Calculated max steps: 1260
Unsloth: Model does not have a default image size - using 512


In [ ]:
trainer_stats = trainer.train(resume_from_checkpoint=resume_from_checkpoint)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 20 | Total steps = 1,260
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 16 x 1) = 32
 "-____-"     Trainable parameters = 43,646,976 of 8,875,725,040 (0.49% trained)
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: alimtegar to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
20,4.209700,4.476460
40,3.103600,3.672106
60,2.782700,3.294928
80,2.556400,3.106545
100,2.434800,3.005637
120,2.378100,2.936720
140,2.264400,2.899930
160,2.155100,2.872082
180,2.207900,2.816737
200,2.124900,2.801531


# Inference

In [ ]:
FastVisionModel.for_inference(model) # Enable for inference

image = None
instruction = val_dataset[0]['messages'][0]['content'][0]['text']

messages = [
    {"role": "user", "content": [
        # {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = False)
outputs = model.generate(**inputs, 
                   streamer = text_streamer, 
                   max_new_tokens = 512,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

In [ ]:
cot_val_sample = cot_val_dataset[0]['messages'][0]['content']
image = cot_val_sample[1]['image']
instruction = cot_val_sample[0]['text']

messages = [
    {"role": "user", "content": [
        {"type" : "image", "image" : image},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = False)
outputs = model.generate(**inputs, 
                   streamer = text_streamer, 
                   max_new_tokens = 1024,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

In [ ]:
# Stop or destroy the vast.ai instance after completing training and inference
# !vastai stop instance 35595876
!vastai destroy instance 35595876